In [22]:
import stormpy

In [23]:
path = './engagement.pm'
prism_program = stormpy.parse_prism_program(path)
model = stormpy.build_model(prism_program)

print(model)

-------------------------------------------------------------- 
Model type: 	DTMC (sparse)
States: 	5
Transitions: 	12
Reward Models:  none
State Labels: 	9 labels
   * deadlock -> 0 item(s)
   * disengaged -> 1 item(s)
   * success -> 1 item(s)
   * converted -> 1 item(s)
   * init -> 1 item(s)
   * failure -> 1 item(s)
   * engaged -> 1 item(s)
   * browsing -> 1 item(s)
   * abandoned -> 1 item(s)
Choice Labels: 	none
-------------------------------------------------------------- 



In [24]:
# Compiling the reachability spec. The spec asks, "from a given state, what is the probability of eventually reaching converted?""
formula_str = """P=? [ F "converted" ]"""
properties = stormpy.parse_properties(formula_str, prism_program)

In [25]:
# Build a model with labels and the state valuations.
options = stormpy.BuilderOptions([p.raw_formula for p in properties])
options.set_build_all_labels()
options.set_build_state_valuations()
model = stormpy.build_sparse_model_with_options(prism_program, options)

labels = model.labeling.get_labels()
print(f"Labels in the model: {labels}")

Labels in the model: {'success', 'abandoned', 'deadlock', 'converted', 'failure', 'browsing', 'engaged', 'init', 'disengaged'}


In [ ]:
result = stormpy.model_checking(model, properties[0])
assert result.result_for_all_states
values = result.get_values()

initial = set(model.initial_states)

print(f"{'idx':>3}  {'labels':<20}  {'P(F \"converted\")':>16}")
print("-" * 54)
for state in model.states:
    names = ", ".join(sorted(model.labeling.get_labels_of_state(state.id)))
    print(f"{state.id:>3}  {names:<20}  {values[state.id]:>16.4f}")

idx  labels                P(F "converted")
------------------------------------------------------
  0  browsing, init                  0.5184
  1  engaged                         0.6720
  2  disengaged                      0.2400
  3  abandoned, failure              0.0000
  4  converted, success              1.0000


In [35]:
values

[0.5184, 0.672, 0.23999999999999994, 0.0, 1.0]